In [1]:
"""
learning_rules_v8_local
=======================
Local version of learning_rules_v8 (converted from Kaggle notebook).
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset, Dataset
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
from pathlib import Path
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import random
from itertools import combinations

OUTPUTS_DIR = Path(r"C:/Users/nilsl/Desktop/Projekte/learning-rules-rsa/outputs")
FMRI_DIR    = Path(r"C:/Users/nilsl/Desktop/Projekte/learning-rules-rsa/outputs_720")
THINGS_DIR  = Path(r"C:/Users/nilsl/Desktop/Projekte/RSA/Datensatz/images_THINGS/object_images")
CIFAR_DIR   = Path(r"C:/Users/nilsl/Desktop/Projekte/learning-rules-rsa/data")
SUBJECTS    = ["sub-01", "sub-02", "sub-03"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

N_EPOCHS = 40
BATCH    = 128
LR       = 1e-3
N_CIFAR  = 8000

IMG_SIZE = 224
N_SEEDS  = 5
SEEDS    = [42, 123, 456, 789, 1337]

C1, C2, C3 = 32, 64, 128
FC1_DIM    = 512
N_CLS      = 10
FEAT_SIZE  = 4
FC1_IN     = C3 * FEAT_SIZE * FEAT_SIZE

T_INF = 10
LR_R  = 0.02
LR_W  = 1e-4

A_P   = 0.003
A_M   = 0.003
TAU_P = 20.0
TAU_M = 20.0
T_SIM = 10

Using device: cpu


In [2]:
print("Checking paths...")
print(f"  FMRI_DIR   exists: {FMRI_DIR.exists()}  -> {FMRI_DIR}")
print(f"  THINGS_DIR exists: {THINGS_DIR.exists()}  -> {THINGS_DIR}")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"  OUTPUTS_DIR ready: {OUTPUTS_DIR}")

Checking paths...
  FMRI_DIR   exists: True  -> C:\Users\nilsl\Desktop\Projekte\learning-rules-rsa\outputs_720
  THINGS_DIR exists: True  -> C:\Users\nilsl\Desktop\Projekte\RSA\Datensatz\images_THINGS\object_images
  OUTPUTS_DIR ready: C:\Users\nilsl\Desktop\Projekte\learning-rules-rsa\outputs


In [3]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("FMRI_DIR exists:", FMRI_DIR.exists())
print("THINGS_DIR exists:", THINGS_DIR.exists())

if FMRI_DIR.exists():
    print("First FMRI files:")
    for p in sorted(FMRI_DIR.iterdir())[:10]:
        print(" ", p.name)

if THINGS_DIR.exists():
    print("First THINGS folders/files:")
    for p in sorted(THINGS_DIR.iterdir())[:10]:
        print(" ", p.name)

FMRI_DIR exists: True
THINGS_DIR exists: True
First FMRI files:
  fmri_rdm_IT_sub-01.npy
  fmri_rdm_IT_sub-02.npy
  fmri_rdm_IT_sub-03.npy
  fmri_rdm_LOC_sub-01.npy
  fmri_rdm_LOC_sub-02.npy
  fmri_rdm_LOC_sub-03.npy
  fmri_rdm_V1_sub-01.npy
  fmri_rdm_V1_sub-02.npy
  fmri_rdm_V1_sub-03.npy
  fmri_rdm_V2_sub-01.npy
First THINGS folders/files:
  aardvark
  abacus
  accordion
  acorn
  air_conditioner
  air_mattress
  air_pump
  airbag
  airboat
  aircraft_carrier


In [4]:
def make_conv_block(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

def _pool_for_fc(c3):
    return F.adaptive_avg_pool2d(c3, FEAT_SIZE)

In [5]:
class Random_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = make_conv_block(3,  C1)
        self.conv2 = make_conv_block(C1, C2)
        self.conv3 = make_conv_block(C2, C3)
        self.fc1   = nn.Linear(FC1_IN, FC1_DIM)
        self.fc2   = nn.Linear(FC1_DIM, N_CLS)

    def forward(self, x):
        x = self.conv3(self.conv2(self.conv1(x)))
        return self.fc2(F.relu(self.fc1(x.view(x.size(0), -1))))

    def get_features(self, x):
        with torch.no_grad():
            c1 = self.conv1(x)
            c2 = self.conv2(c1)
            c3 = self.conv3(c2)
            h1 = F.relu(self.fc1(_pool_for_fc(c3).view(c3.size(0), -1)))
        return c1.mean([2,3]), c2.mean([2,3]), c3.mean([2,3]), h1


def train_random(loader):
    model = Random_CNN().to(DEVICE)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            correct += (out.argmax(1) == y).sum().item()
            total   += y.size(0)
    print(f"    Random baseline accuracy: {correct/total:.3f} (expected ~0.1)")
    return model, [{"epoch": 0, "loss": float('nan'), "acc": correct / total}]

In [6]:
class BP_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = make_conv_block(3,  C1)
        self.conv2 = make_conv_block(C1, C2)
        self.conv3 = make_conv_block(C2, C3)
        self.fc1   = nn.Linear(FC1_IN, FC1_DIM)
        self.fc2   = nn.Linear(FC1_DIM, N_CLS)
        self.drop  = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv3(self.conv2(self.conv1(x)))
        return self.fc2(self.drop(F.relu(self.fc1(x.view(x.size(0), -1)))))

    def get_features(self, x):
        with torch.no_grad():
            c1 = self.conv1(x)
            c2 = self.conv2(c1)
            c3 = self.conv3(c2)
            h1 = F.relu(self.fc1(_pool_for_fc(c3).view(c3.size(0), -1)))
        return c1.mean([2,3]), c2.mean([2,3]), c3.mean([2,3]), h1


def train_bp(loader):
    model = BP_CNN().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, N_EPOCHS)
    hist  = []
    for epoch in range(N_EPOCHS):
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            with torch.no_grad():
                tl += loss.item()
                tc += (out.argmax(1) == y).sum().item()
                tn += y.size(0)
        sched.step()
        hist.append({"epoch": epoch+1, "loss": tl/len(loader), "acc": tc/tn})
        if (epoch+1) % 10 == 0:
            print(f"    Epoch {epoch+1:3d}: loss={tl/len(loader):.4f}  acc={tc/tn:.3f}")
    model.eval()
    return model, hist

In [7]:
class FAConvFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, W, b, B_feedback, stride, padding):
        ctx.save_for_backward(x, W, b, B_feedback)
        ctx.stride = stride
        ctx.padding = padding
        return F.conv2d(x, W, b, stride=stride, padding=padding)

    @staticmethod
    def backward(ctx, grad_output):
        x, W, b, B = ctx.saved_tensors
        stride, padding = ctx.stride, ctx.padding
        grad_W = torch.nn.grad.conv2d_weight(
            x, W.shape, grad_output, stride=stride, padding=padding
        )
        grad_b = grad_output.sum([0,2,3]) if b is not None else None
        grad_x = F.conv_transpose2d(grad_output, B, stride=stride, padding=padding)
        return grad_x, grad_W, grad_b, None, None, None


class FAConv2d(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.W = nn.Parameter(
            nn.init.kaiming_normal_(torch.empty(out_c, in_c, kernel_size, kernel_size))
        )
        self.b = nn.Parameter(torch.zeros(out_c))
        B = nn.init.xavier_normal_(torch.randn(out_c, in_c, kernel_size, kernel_size))
        self.register_buffer("B_feedback", B)
        self.stride = stride
        self.padding = padding

    def forward(self, x):
        return FAConvFunction.apply(
            x, self.W, self.b, self.B_feedback, self.stride, self.padding
        )


def make_fa_conv_block(in_c, out_c):
    return nn.Sequential(
        FAConv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class FA_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = make_fa_conv_block(3,  C1)
        self.conv2 = make_fa_conv_block(C1, C2)
        self.conv3 = make_fa_conv_block(C2, C3)
        self.fc1   = nn.Linear(FC1_IN, FC1_DIM)
        self.fc2   = nn.Linear(FC1_DIM, N_CLS)
        self.drop  = nn.Dropout(0.3)

    def forward(self, x):
        x = self.conv3(self.conv2(self.conv1(x)))
        return self.fc2(self.drop(F.relu(self.fc1(x.view(x.size(0), -1)))))

    def get_features(self, x):
        with torch.no_grad():
            c1 = self.conv1(x)
            c2 = self.conv2(c1)
            c3 = self.conv3(c2)
            h1 = F.relu(self.fc1(_pool_for_fc(c3).view(c3.size(0), -1)))
        return c1.mean([2,3]), c2.mean([2,3]), c3.mean([2,3]), h1


def train_fa(loader):
    model = FA_CNN().to(DEVICE)
    opt   = torch.optim.SGD(model.parameters(), lr=LR*0.5, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, N_EPOCHS)
    hist  = []
    for epoch in range(N_EPOCHS):
        model.train()
        tl, tc, tn = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl += loss.item()
            tc += (out.argmax(1) == y).sum().item()
            tn += y.size(0)
        sched.step()
        hist.append({"epoch": epoch+1, "loss": tl/len(loader), "acc": tc/tn})
        if (epoch+1) % 10 == 0:
            print(f"    Epoch {epoch+1:3d}: loss={tl/len(loader):.4f}  acc={tc/tn:.3f}")
    model.eval()
    return model, hist

In [8]:
class PC_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.W1 = nn.Conv2d(3,  C1, 3, padding=1, bias=False)
        self.W2 = nn.Conv2d(C1, C2, 3, padding=1, bias=False)
        self.W3 = nn.Conv2d(C2, C3, 3, padding=1, bias=False)
        self.pool = nn.MaxPool2d(2)
        self.bn1  = nn.BatchNorm2d(C1)
        self.bn2  = nn.BatchNorm2d(C2)
        self.bn3  = nn.BatchNorm2d(C3)
        self.P2   = nn.ConvTranspose2d(C2, C1, 3, padding=1, bias=False)
        self.P3   = nn.ConvTranspose2d(C3, C2, 3, padding=1, bias=False)
        self.fc1  = nn.Linear(FC1_IN, FC1_DIM)
        self.fc2  = nn.Linear(FC1_DIM, N_CLS)
        self.clf_opt = None

    def _make_opt(self):
        self.clf_opt = torch.optim.Adam(
            list(self.fc1.parameters()) + list(self.fc2.parameters()), lr=LR
        )

    def infer(self, x):
        with torch.no_grad():
            r1 = self.pool(F.relu(self.bn1(self.W1(x))))
            r2 = self.pool(F.relu(self.bn2(self.W2(r1))))
            r3 = self.pool(F.relu(self.bn3(self.W3(r2))))
            for _ in range(T_INF):
                pred1 = torch.tanh(F.interpolate(self.P2(r2), size=r1.shape[2:], mode="nearest"))
                pred2 = torch.tanh(F.interpolate(self.P3(r3), size=r2.shape[2:], mode="nearest"))
                e1 = r1 - pred1
                e2 = r2 - pred2
                r1 = F.relu(r1 + LR_R * (-e1))
                r2 = F.relu(r2 + LR_R * (-e2 + F.avg_pool2d(F.conv2d(e1, self.W2.weight, padding=1), 2)))
                r3 = F.relu(r3 + LR_R * F.avg_pool2d(F.conv2d(e2, self.W3.weight, padding=1), 2))
        return r1, r2, r3

    def weight_update(self, x, r1, r2, r3):
        with torch.no_grad():
            r1i = self.pool(F.relu(self.bn1(self.W1(x))))
            r2i = self.pool(F.relu(self.bn2(self.W2(r1i))))
            e1 = (r1 - r1i).clamp(-0.5, 0.5)
            e2 = (r2 - r2i).clamp(-0.5, 0.5)

            dW1 = (
                e1.mean([0,2,3]).unsqueeze(1) * x.mean([0,2,3]).unsqueeze(0)
            ).unsqueeze(-1).unsqueeze(-1).expand_as(self.W1.weight).clamp(-0.01, 0.01)
            self.W1.weight.data += LR_W * dW1

            dW2 = (
                e2.mean([0,2,3]).unsqueeze(1) * r1.mean([0,2,3]).unsqueeze(0)
            ).unsqueeze(-1).unsqueeze(-1).expand_as(self.W2.weight).clamp(-0.01, 0.01)
            self.W2.weight.data += LR_W * dW2

    def get_features(self, x):
        with torch.no_grad():
            r1, r2, r3 = self.infer(x)
            h1 = F.relu(self.fc1(_pool_for_fc(r3).view(r3.size(0), -1)))
        return r1.mean([2,3]), r2.mean([2,3]), r3.mean([2,3]), h1

    def step(self, x, y):
        r1, r2, r3 = self.infer(x)
        self.weight_update(x, r1, r2, r3)
        self.clf_opt.zero_grad()
        logit = self.fc2(F.relu(self.fc1(r3.detach().view(r3.size(0), -1))))
        loss = F.cross_entropy(logit, y)
        loss.backward()
        self.clf_opt.step()
        return loss.item(), (logit.argmax(1) == y).float().mean().item()

    def eval(self):
        pass


def train_pc(loader):
    model = PC_CNN().to(DEVICE)
    model._make_opt()
    hist = []
    for epoch in range(N_EPOCHS):
        tl, ta, n = 0.0, 0.0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss, acc = model.step(x, y)
            tl += loss
            ta += acc
            n += 1
        hist.append({"epoch": epoch+1, "loss": tl/n, "acc": ta/n})
        if (epoch+1) % 10 == 0:
            print(f"    Epoch {epoch+1:3d}: loss={tl/n:.4f}  acc={ta/n:.3f}")
    return model, hist

In [9]:
class STDP_Conv:
    def __init__(self, in_c, out_c, kernel_size=3, padding=1):
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, padding=padding, bias=False)
        nn.init.kaiming_normal_(self.conv.weight)

    def poisson_spikes(self, rates):
        r = rates.clamp(0, 1).unsqueeze(-1).expand(*rates.shape, T_SIM)
        return (torch.rand_like(r) < r / T_SIM).float()

    def first_spike(self, spikes):
        has = spikes.any(-1)
        t = torch.argmax(spikes, dim=-1).float()
        t[~has] = float(T_SIM + 1)
        return t

    def stdp_update(self, pre_act, post_act, lr=5e-4):
        with torch.no_grad():
            pre_s = self.poisson_spikes(torch.sigmoid(pre_act))
            post_s = self.poisson_spikes(torch.sigmoid(post_act))
            t_pre = self.first_spike(pre_s)
            t_post = self.first_spike(post_s)
            dt = t_post.unsqueeze(1) - t_pre.unsqueeze(2)
            dW = (
                A_P * torch.exp(-dt.clamp(min=0) / TAU_P)
                - A_M * torch.exp(dt.clamp(max=0) / TAU_M)
            ).mean(0).clamp(-0.002, 0.002)
            dW_conv = dW.T.view(
                self.conv.weight.size(0), self.conv.weight.size(1), 1, 1
            ).expand_as(self.conv.weight)
            self.conv.weight.data += lr * dW_conv
            self.conv.weight.data.clamp_(-1.0, 1.0)

    def forward(self, x, do_stdp=False, pre_act=None):
        out = F.relu(self.conv(x))
        if do_stdp and pre_act is not None:
            self.stdp_update(pre_act, out.mean([2, 3]))
        return out


class STDP_CNN:
    def __init__(self):
        self.L1 = STDP_Conv(3,  C1)
        self.L2 = STDP_Conv(C1, C2)
        self.L3 = STDP_Conv(C2, C3)
        self.pool = nn.MaxPool2d(2)
        self.bn1 = nn.BatchNorm2d(C1)
        self.bn2 = nn.BatchNorm2d(C2)
        self.bn3 = nn.BatchNorm2d(C3)
        self.fc1 = nn.Linear(FC1_IN, FC1_DIM)
        self.fc2 = nn.Linear(FC1_DIM, N_CLS)
        self.clf_opt = None

    def to_device(self, device):
        self.L1.conv = self.L1.conv.to(device)
        self.L2.conv = self.L2.conv.to(device)
        self.L3.conv = self.L3.conv.to(device)
        self.pool = self.pool.to(device)
        self.bn1 = self.bn1.to(device)
        self.bn2 = self.bn2.to(device)
        self.bn3 = self.bn3.to(device)
        self.fc1 = self.fc1.to(device)
        self.fc2 = self.fc2.to(device)
        self.clf_opt = torch.optim.Adam(
            list(self.fc1.parameters()) + list(self.fc2.parameters()) +
            [self.bn1.weight, self.bn1.bias, self.bn2.weight, self.bn2.bias, self.bn3.weight, self.bn3.bias],
            lr=LR
        )
        return self

    def _forward(self, x, do_stdp=False):
        c1 = self.pool(F.relu(self.bn1(self.L1.forward(x,  do_stdp, x.mean([2,3])))))
        c2 = self.pool(F.relu(self.bn2(self.L2.forward(c1, do_stdp, c1.mean([2,3])))))
        c3 = self.pool(F.relu(self.bn3(self.L3.forward(c2, do_stdp, c2.mean([2,3])))))
        return c1, c2, c3

    def get_features(self, x):
        with torch.no_grad():
            c1, c2, c3 = self._forward(x, do_stdp=False)
            h1 = F.relu(self.fc1(_pool_for_fc(c3).view(c3.size(0), -1)))
        return c1.mean([2,3]), c2.mean([2,3]), c3.mean([2,3]), h1

    def step(self, x, y):
        self._forward(x, do_stdp=True)
        self.clf_opt.zero_grad()
        _, _, c3b = self._forward(x, do_stdp=False)
        logit = self.fc2(F.relu(self.fc1(c3b.view(c3b.size(0), -1))))
        loss = F.cross_entropy(logit, y)
        loss.backward()
        self.clf_opt.step()
        acc = (logit.argmax(1) == y).float().mean().item()
        return loss.detach().item(), acc

    def eval(self):
        pass


def train_stdp(loader):
    model = STDP_CNN().to_device(DEVICE)
    hist = []
    for epoch in range(N_EPOCHS):
        tl, ta, n = 0.0, 0.0, 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss, acc = model.step(x, y)
            tl += loss
            ta += acc
            n += 1
        hist.append({"epoch": epoch+1, "loss": tl/n, "acc": ta/n})
        if (epoch+1) % 10 == 0:
            print(f"    Epoch {epoch+1:3d}: loss={tl/n:.4f}  acc={ta/n:.3f}")
    return model, hist

In [10]:
def compute_rdm(f):
    f = f.detach().cpu().numpy() if torch.is_tensor(f) else np.array(f)
    return squareform(pdist(f, metric="correlation"))

def rsa_score(a, b):
    n = min(a.shape[0], b.shape[0])
    idx = np.triu_indices(n, k=1)
    r, p = spearmanr(a[:n, :n][idx], b[:n, :n][idx])
    return float(r), float(p)

def bootstrap_ci(rdm_a, rdm_b, n_boot=500, ci=0.95, seed=42):
    n = min(rdm_a.shape[0], rdm_b.shape[0])
    idx = np.triu_indices(n, k=1)
    va = rdm_a[:n, :n][idx]
    vb = rdm_b[:n, :n][idx]
    rng = np.random.default_rng(seed)
    boot_rs = []
    for _ in range(n_boot):
        s = rng.integers(0, len(va), len(va))
        boot_rs.append(spearmanr(va[s], vb[s])[0])
    return (
        float(np.percentile(boot_rs, (1-ci)/2*100)),
        float(np.percentile(boot_rs, (1+ci)/2*100)),
        boot_rs
    )

def load_fmri_rdm(roi, sub):
    p = FMRI_DIR / f"fmri_rdm_{roi}_{sub}.npy"
    return np.load(str(p)) if p.exists() else None

def noise_ceiling(sub_ids, roi, n_splits=200):
    rdms = [r for r in [load_fmri_rdm(roi, s) for s in sub_ids] if r is not None]
    if len(rdms) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(42)
    n = rdms[0].shape[0]
    idx = np.triu_indices(n, k=1)
    rhos = []
    for _ in range(n_splits):
        perm = rng.permutation(len(rdms))
        half1 = np.mean([rdms[i][idx] for i in perm[:len(rdms)//2]], 0)
        half2 = np.mean([rdms[i][idx] for i in perm[len(rdms)//2:]], 0)
        r, _ = spearmanr(half1, half2)
        rhos.append(2*r / (1+r) if r < 1 else 1.0)
    return float(np.percentile(rhos, 2.5)), float(np.mean(rhos))

def load_stim_order(sub="sub-01"):
    p = FMRI_DIR / f"stim_order_{sub}.txt"
    with open(p) as f:
        return [l.strip() for l in f if l.strip()]

def find_img(stimulus):
    name = stimulus.replace(".jpg", "")
    hits = list(THINGS_DIR.rglob(f"{name}.jpg"))
    if hits:
        return hits[0]

    parts = name.split("_")
    last = parts[-1]
    concept = "_".join(parts[:-1]) if (len(parts) > 1 and len(last) <= 4 and any(c.isdigit() for c in last)) else name

    hits = list(THINGS_DIR.rglob(f"{concept}*.jpg"))
    if hits:
        return sorted(hits)[0]

    return None

class ImgDS(Dataset):
    def __init__(self, paths, t):
        self.paths, self.t = paths, t

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.t(Image.open(self.paths[i]).convert("RGB")), i

def extract_features(model, paths, transform):
    loader = DataLoader(ImgDS(paths, transform), batch_size=64, shuffle=False, num_workers=0)

    c1s, c2s, c3s, h1s = [], [], [], []
    model.eval()
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            imgs = imgs.to(DEVICE)
            c1, c2, c3, h1 = model.get_features(imgs)

            def np_(t):
                return t.cpu().numpy() if torch.is_tensor(t) else np.array(t)

            c1s.append(np_(c1))
            c2s.append(np_(c2))
            c3s.append(np_(c3))
            h1s.append(np_(h1))

    if len(c1s) == 0:
        raise ValueError("No features extracted: paths or loader is empty")

    return (
        np.concatenate(c1s),
        np.concatenate(c2s),
        np.concatenate(c3s),
        np.concatenate(h1s),
    )

LAYER_ROI_FIXED = {
    "Conv1": (0, ["V1", "V2"]),
    "Conv2": (1, ["V1", "V2"]),
    "Conv3": (2, ["LOC"]),
    "FC1":   (3, ["IT"]),
}

LAYER_ROI_ALL = {
    l: (i, ["V1", "V2", "LOC", "IT"])
    for i, l in enumerate(["Conv1", "Conv2", "Conv3", "FC1"])
}

def run_rsa(model, paths, transform, rule, layer_roi_map=None):
    if layer_roi_map is None:
        layer_roi_map = LAYER_ROI_FIXED
    print(f"\n  RSA: {rule}")
    feats = extract_features(model, paths, transform)
    rdms = [compute_rdm(f) for f in feats]
    rows = []

    for layer_name, (feat_idx, rois) in layer_roi_map.items():
        for roi in rois:
            sub_scores = [
                {"sub": s, "rho": rsa_score(rdms[feat_idx], load_fmri_rdm(roi, s))[0]}
                for s in SUBJECTS if load_fmri_rdm(roi, s) is not None
            ]
            if not sub_scores:
                continue

            brain_mean = np.mean(
                [load_fmri_rdm(roi, s) for s in SUBJECTS if load_fmri_rdm(roi, s) is not None],
                axis=0
            )
            n = min(rdms[feat_idx].shape[0], brain_mean.shape[0])
            rho_mb, _ = rsa_score(rdms[feat_idx][:n, :n], brain_mean[:n, :n])
            lo, hi, _ = bootstrap_ci(rdms[feat_idx][:n, :n], brain_mean[:n, :n])

            rows.append({
                "rule": rule,
                "layer": layer_name,
                "roi": roi,
                "rho": float(rho_mb),
                "ci_lo": lo,
                "ci_hi": hi,
                "n_subs": len(sub_scores),
                "rho_sub01": next((s["rho"] for s in sub_scores if s["sub"] == "sub-01"), np.nan),
                "rho_sub02": next((s["rho"] for s in sub_scores if s["sub"] == "sub-02"), np.nan),
                "rho_sub03": next((s["rho"] for s in sub_scores if s["sub"] == "sub-03"), np.nan),
                "rho_sub_mean": float(np.mean([s["rho"] for s in sub_scores])),
            })
            print(f"    {layer_name:5s} vs {roi:3s}: ρ={rho_mb:.4f} [{lo:.4f},{hi:.4f}]")

    return rows, rdms

In [11]:
def benjamini_hochberg(p_values):
    n = len(p_values)
    sorted_idx = np.argsort(p_values)
    sorted_p = np.array(p_values)[sorted_idx]
    adjusted = np.zeros(n)
    cum_min = 1.0
    for i in range(n-1, -1, -1):
        cum_min = min(cum_min, sorted_p[i] * n / (i + 1))
        adjusted[sorted_idx[i]] = min(cum_min, 1.0)
    return adjusted.tolist()

def permutation_test(rdm_a, rdm_b, brain_rdm, n_perm=1000):
    rng = np.random.default_rng(42)
    n = min(rdm_a.shape[0], rdm_b.shape[0], brain_rdm.shape[0])
    idx = np.triu_indices(n, k=1)
    va = rdm_a[:n, :n][idx]
    vb = rdm_b[:n, :n][idx]
    vbr = brain_rdm[:n, :n][idx]
    observed = float(spearmanr(va, vbr)[0]) - float(spearmanr(vb, vbr)[0])
    null = []
    for _ in range(n_perm):
        p1 = rng.permutation(len(vbr))
        p2 = rng.permutation(len(vbr))
        null.append(float(spearmanr(va, vbr[p1])[0]) - float(spearmanr(vb, vbr[p2])[0]))
    return observed, float(np.mean(np.abs(null) >= np.abs(observed)))

def run_all_stats(model_rdms, rsa_df):
    rules = [r for r in ["Random Weights", "Backprop", "Feedback Alignment", "Predictive Coding", "STDP"] if r in model_rdms]
    layer_map = {"V1": "Conv1", "V2": "Conv1", "LOC": "Conv3", "IT": "FC1"}
    perm_rows, all_p = [], []

    for roi in ["V1", "V2", "LOC", "IT"]:
        valid = [load_fmri_rdm(roi, s) for s in SUBJECTS if load_fmri_rdm(roi, s) is not None]
        if len(valid) == 0:
            continue
        brain = np.mean(valid, axis=0)
        layer = layer_map[roi]
        print(f"\n  Permutation tests — {roi} ({layer}):")

        for rule_a, rule_b in combinations(rules, 2):
            if layer not in model_rdms.get(rule_a, {}) or layer not in model_rdms.get(rule_b, {}):
                continue
            delta, p = permutation_test(model_rdms[rule_a][layer], model_rdms[rule_b][layer], brain)
            perm_rows.append({
                "roi": roi,
                "layer": layer,
                "rule_a": rule_a,
                "rule_b": rule_b,
                "delta": round(delta, 5),
                "p_uncorrected": round(p, 4)
            })
            all_p.append(p)
            print(
                f"    {rule_a:25s} vs {rule_b:25s}: Δ={delta:+.5f}  p={p:.4f} "
                f"{'***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'}"
            )

    if all_p:
        fdr = benjamini_hochberg(all_p)
        for i, row in enumerate(perm_rows):
            row["p_fdr"] = round(fdr[i], 4)
            row["sig_fdr"] = "***" if fdr[i] < 0.001 else "**" if fdr[i] < 0.01 else "*" if fdr[i] < 0.05 else "ns"

    return pd.DataFrame(perm_rows)

def subject_level_analysis(rsa_df):
    print("\n" + "="*55 + "\nSUBJECT-LEVEL ANALYSIS\n" + "="*55)
    rows = []
    for roi in ["V1", "V2", "LOC", "IT"]:
        sub_df = rsa_df[rsa_df["roi"] == roi].copy()
        if sub_df.empty:
            continue
        best = sub_df.loc[sub_df.groupby("rule")["rho"].idxmax()]
        print(f"\n  {roi}:")
        for col, name in [("rho_sub01", "sub-01"), ("rho_sub02", "sub-02"), ("rho_sub03", "sub-03")]:
            if col not in best.columns:
                continue
            ranking = best[["rule", col]].dropna().sort_values(col, ascending=False)
            print(f"    {name}: {' > '.join(ranking['rule'].tolist())}")
            for _, row in ranking.iterrows():
                rows.append({"roi": roi, "subject": name, "rule": row["rule"], "rho": round(row[col], 4)})
    return pd.DataFrame(rows)

def aggregate_seed_results(seed_df):
    agg = []
    for (rule, layer, roi), g in seed_df.groupby(["rule", "layer", "roi"]):
        agg.append({
            "rule": rule,
            "layer": layer,
            "roi": roi,
            "rho": round(g["rho"].mean(), 5),
            "rho_std": round(g["rho"].std(), 5),
            "ci_lo": round(g["ci_lo"].mean(), 5),
            "ci_hi": round(g["ci_hi"].mean(), 5),
            "n_seeds": len(g),
            "n_subs": int(g["n_subs"].iloc[0]),
            "rho_sub01": round(g["rho_sub01"].mean(), 5),
            "rho_sub02": round(g["rho_sub02"].mean(), 5),
            "rho_sub03": round(g["rho_sub03"].mean(), 5),
            "rho_sub_mean": round(g["rho_sub_mean"].mean(), 5),
        })
    return pd.DataFrame(agg)

COLORS = {
    "Random Weights": "#999999",
    "Backprop": "#2E86AB",
    "Feedback Alignment": "#E84855",
    "Predictive Coding": "#3BB273",
    "STDP": "#F4A261"
}

In [12]:
def plot_rsa(df, nc_dict=None):
    rois = [r for r in ["V1", "V2", "LOC", "IT"] if r in df["roi"].values]
    rules = [r for r in COLORS if r in df["rule"].values]
    x = np.arange(len(rois))
    w = 0.7 / len(rules)
    offs = np.linspace(-(len(rules)-1)*w/2, (len(rules)-1)*w/2, len(rules))
    _, ax = plt.subplots(figsize=(13, 5.5))

    if nc_dict:
        for i, roi in enumerate(rois):
            if roi in nc_dict:
                ax.fill_between([i-0.4, i+0.4], *nc_dict[roi], alpha=0.12, color="gray", zorder=1)

    for i, rule in enumerate(rules):
        grp = df[df["rule"] == rule].groupby("roi")
        for j, roi in enumerate(rois):
            if roi not in grp.groups:
                continue
            row = grp.get_group(roi).iloc[0]
            rho = row["rho"]
            ax.bar(j + offs[i], rho, w*0.9, color=COLORS[rule], alpha=0.85, label=rule if j == 0 else "")
            yerr = row.get("rho_std", None)
            if yerr is not None and not np.isnan(yerr):
                ax.errorbar(j + offs[i], rho, yerr=yerr, fmt="none", color="black", capsize=3, lw=1.5, zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(rois, fontsize=12)
    ax.set_ylabel("Spearman ρ (model–brain)", fontsize=12)
    ax.set_title("Which learning rule produces the most brain-like representations?", fontsize=13)
    ax.axhline(0, color="black", lw=0.5)
    handles, labels = ax.get_legend_handles_labels()
    seen = dict(zip(labels, handles))
    ax.legend(seen.values(), seen.keys(), fontsize=9, ncol=3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / "rsa_comparison_cnn.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print("  rsa_comparison_cnn.png saved")

def plot_seed_variability(seed_df):
    layer_map = {"V1": "Conv1", "V2": "Conv1", "LOC": "Conv3", "IT": "FC1"}
    rois = ["V1", "V2", "LOC", "IT"]
    rules = [r for r in COLORS if r in seed_df["rule"].values]
    x = np.arange(len(rois))
    w = 0.7 / len(rules)
    offs = np.linspace(-(len(rules)-1)*w/2, (len(rules)-1)*w/2, len(rules))
    _, ax = plt.subplots(figsize=(12, 5))

    for i, rule in enumerate(rules):
        means, stds = [], []
        for roi in rois:
            g = seed_df[
                (seed_df["rule"] == rule) &
                (seed_df["roi"] == roi) &
                (seed_df["layer"] == layer_map[roi])
            ]["rho"]
            means.append(g.mean() if len(g) else np.nan)
            stds.append(g.std() if len(g) > 1 else 0.0)

        ax.bar(
            x + offs[i], means, w*0.9, color=COLORS[rule], alpha=0.8,
            label=rule, yerr=stds, capsize=3, error_kw={"lw": 1, "color": "black"}
        )

    ax.set_xticks(x)
    ax.set_xticklabels(rois, fontsize=12)
    ax.set_ylabel("Spearman ρ (mean ± std across seeds)", fontsize=12)
    ax.set_title(f"Seed variability ({N_SEEDS} seeds, {IMG_SIZE}×{IMG_SIZE} THINGS input)", fontsize=12)
    ax.axhline(0, color="black", lw=0.5)
    ax.legend(fontsize=9, ncol=2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / "seed_variability.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print("  seed_variability.png saved")

In [13]:
import collections

print("Mapping images... (dies passiert nur einmal)")
# Einmaliges Durchsuchen und Speichern im Memory
img_map = {p.name: p for p in THINGS_DIR.rglob("*.jpg")}
print(f"Mapping fertig: {len(img_map)} Bilder gefunden.")

def find_img_fast(stimulus):
    name = stimulus.replace(".jpg", "") + ".jpg"
    return img_map.get(name)

# Jetzt der Aufruf
stimuli = load_stim_order("sub-01")
paths = [p for p in [find_img_fast(s) for s in stimuli] if p is not None]
print(f"DEBUG: {len(paths)}/{len(stimuli)} Bilder gefunden.")

Mapping images... (dies passiert nur einmal)
Mapping fertig: 26107 Bilder gefunden.
DEBUG: 720/720 Bilder gefunden.


In [14]:
print("Loading CIFAR-10...")
tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

full = torchvision.datasets.CIFAR10(str(CIFAR_DIR), train=True, download=True, transform=tf)
idx = torch.randperm(len(full))[:N_CIFAR].tolist()
train_loader = DataLoader(Subset(full, idx), batch_size=BATCH, shuffle=True, num_workers=0, drop_last=True)
print(f"  {N_CIFAR} samples, {len(train_loader)} batches/epoch")

tf_things = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])

Loading CIFAR-10...
  8000 samples, 62 batches/epoch


In [15]:
print("Computing noise ceilings...")
nc_dict = {}
for roi in ["V1", "V2", "LOC", "IT"]:
    lo, hi = noise_ceiling(SUBJECTS, roi)
    nc_dict[roi] = (lo, hi)
    if not np.isnan(lo):
        print(f"  {roi}: NC [{lo:.3f}, {hi:.3f}]")

Computing noise ceilings...
  V1: NC [0.036, 0.106]
  V2: NC [0.019, 0.062]
  LOC: NC [0.022, 0.065]
  IT: NC [0.022, 0.066]


In [16]:
TRAIN_FNS = {
    "Random Weights": train_random,
    "Backprop": train_bp,
    "Feedback Alignment": train_fa,
    "Predictive Coding": train_pc,
    "STDP": train_stdp,
}

In [ ]:
seed_rows = []
seed_rdms = {
    rule: {layer: [] for layer in ["Conv1", "Conv2", "Conv3", "FC1"]}
    for rule in TRAIN_FNS
}

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*55}\nSEED {seed_idx+1}/{N_SEEDS}  (seed={seed})\n{'='*55}")
    rdm_dir_seed = OUTPUTS_DIR / "model_rdms" / f"seed_{seed_idx}"
    rdm_dir_seed.mkdir(parents=True, exist_ok=True)

    histories = {}
    for rule, fn in TRAIN_FNS.items():
        print(f"\n  {'Training' if rule != 'Random Weights' else 'Initializing'}: {rule}")
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        model, hist = fn(train_loader)
        histories[rule] = hist

        rows, rdms = run_rsa(model, paths, tf_things, rule)
        for row in rows:
            row["seed"] = seed
            row["seed_idx"] = seed_idx
        seed_rows.extend(rows)

        rule_key = rule.lower().replace(" ", "_")
        for layer_name, rdm in zip(["Conv1", "Conv2", "Conv3", "FC1"], rdms):
            np.save(str(rdm_dir_seed / f"rdm_{rule_key}_{layer_name}.npy"), rdm)
            seed_rdms[rule][layer_name].append(rdm)

        if seed_idx == 0:
            if hasattr(model, "state_dict"):
                torch.save(model.state_dict(), str(OUTPUTS_DIR / f"model_weights_{rule_key}.pt"))
            elif hasattr(model, "L1"):  # STDP_CNN custom class
                stdp_weights = {
                    "conv1.0.W": torch.tensor(model.L1.W),
                    "conv2.0.W": torch.tensor(model.L2.W),
                    "conv3.0.W": torch.tensor(model.L3.W),
                }
                torch.save(stdp_weights, str(OUTPUTS_DIR / f"model_weights_{rule_key}.pt"))
        del model
        torch.cuda.empty_cache()


SEED 1/5  (seed=42)

  Initializing: Random Weights
    Random baseline accuracy: 0.100 (expected ~0.1)

  RSA: Random Weights
    Conv1 vs V1 : ρ=0.0840 [0.0804,0.0881]
    Conv1 vs V2 : ρ=0.0495 [0.0459,0.0535]
    Conv2 vs V1 : ρ=0.0916 [0.0880,0.0956]
    Conv2 vs V2 : ρ=0.0567 [0.0530,0.0607]
    Conv3 vs LOC: ρ=-0.0079 [-0.0121,-0.0040]
    FC1   vs IT : ρ=0.0067 [0.0029,0.0106]

  Training: Backprop
    Epoch  10: loss=0.9931  acc=0.642
    Epoch  20: loss=0.7475  acc=0.732
    Epoch  30: loss=0.5901  acc=0.794
    Epoch  40: loss=0.5423  acc=0.810

  RSA: Backprop
    Conv1 vs V1 : ρ=0.0252 [0.0210,0.0290]
    Conv1 vs V2 : ρ=0.0133 [0.0099,0.0173]
    Conv2 vs V1 : ρ=-0.0269 [-0.0307,-0.0231]
    Conv2 vs V2 : ρ=-0.0195 [-0.0233,-0.0158]
    Conv3 vs LOC: ρ=0.0113 [0.0074,0.0153]
    FC1   vs IT : ρ=0.0139 [0.0098,0.0179]

  Training: Feedback Alignment
    Epoch  10: loss=1.9097  acc=0.322
    Epoch  20: loss=1.8115  acc=0.362
    Epoch  30: loss=1.7857  acc=0.365
    Epoch 

In [18]:
print("\n" + "="*55 + "\nAGGREGATING ACROSS SEEDS\n" + "="*55)
rdm_dir = OUTPUTS_DIR / "model_rdms"
rdm_dir.mkdir(exist_ok=True)

model_rdms_mean = {}
for rule in TRAIN_FNS:
    rule_key = rule.lower().replace(" ", "_")
    model_rdms_mean[rule] = {}
    for layer_name in ["Conv1", "Conv2", "Conv3", "FC1"]:
        mean_rdm = np.mean(seed_rdms[rule][layer_name], axis=0)
        np.save(str(rdm_dir / f"rdm_{rule_key}_{layer_name}.npy"), mean_rdm)
        model_rdms_mean[rule][layer_name] = mean_rdm

print("  Mean RDMs saved.")

seed_df = pd.DataFrame(seed_rows)
seed_df.to_csv(str(OUTPUTS_DIR / "rsa_results_seeds.csv"), index=False)

df = aggregate_seed_results(seed_df)
df.to_csv(str(OUTPUTS_DIR / "rsa_results_cnn.csv"), index=False)

print(f"  rsa_results_seeds.csv: {len(seed_df)} rows")
print(f"  rsa_results_cnn.csv:   {len(df)} rows (mean across seeds)")


AGGREGATING ACROSS SEEDS
  Mean RDMs saved.
  rsa_results_seeds.csv: 150 rows
  rsa_results_cnn.csv:   30 rows (mean across seeds)


In [19]:
all_rows_full = []
for rule in TRAIN_FNS:
    rule_key = rule.lower().replace(" ", "_")
    rdms_list = [np.load(str(rdm_dir / f"rdm_{rule_key}_{ln}.npy")) for ln in ["Conv1", "Conv2", "Conv3", "FC1"]]

    for layer_name, (feat_idx, rois) in LAYER_ROI_ALL.items():
        for roi in rois:
            valid = [load_fmri_rdm(roi, s) for s in SUBJECTS if load_fmri_rdm(roi, s) is not None]
            if len(valid) == 0:
                continue
            brain = np.mean(valid, axis=0)
            n = min(rdms_list[feat_idx].shape[0], brain.shape[0])
            r, _ = rsa_score(rdms_list[feat_idx][:n, :n], brain[:n, :n])
            all_rows_full.append({
                "rule": rule,
                "layer": layer_name,
                "roi": roi,
                "rho": round(r, 4)
            })

pd.DataFrame(all_rows_full).to_csv(str(OUTPUTS_DIR / "rsa_all_layers_all_rois.csv"), index=False)

In [20]:
perm_df = run_all_stats(model_rdms_mean, df)
perm_df.to_csv(str(OUTPUTS_DIR / "permutation_results_fdr.csv"), index=False)

sub_df = subject_level_analysis(df)
sub_df.to_csv(str(OUTPUTS_DIR / "subject_level_rsa.csv"), index=False)


  Permutation tests — V1 (Conv1):
    Random Weights            vs Backprop                 : Δ=+0.04374  p=0.0000 ***
    Random Weights            vs Feedback Alignment       : Δ=+0.06509  p=0.0000 ***
    Random Weights            vs Predictive Coding        : Δ=+0.01962  p=0.0000 ***
    Random Weights            vs STDP                     : Δ=+0.01101  p=0.0000 ***
    Backprop                  vs Feedback Alignment       : Δ=+0.02135  p=0.0000 ***
    Backprop                  vs Predictive Coding        : Δ=-0.02412  p=0.0000 ***
    Backprop                  vs STDP                     : Δ=-0.03273  p=0.0000 ***
    Feedback Alignment        vs Predictive Coding        : Δ=-0.04547  p=0.0000 ***
    Feedback Alignment        vs STDP                     : Δ=-0.05408  p=0.0000 ***
    Predictive Coding         vs STDP                     : Δ=-0.00861  p=0.0020 **

  Permutation tests — V2 (Conv1):
    Random Weights            vs Backprop                 : Δ=+0.02586  p=0.0000 

In [21]:
print("\nGenerating plots...")
plot_rsa(df, nc_dict)
plot_seed_variability(seed_df)

if not perm_df.empty:
    rois_p = perm_df["roi"].unique()
    _, axes = plt.subplots(1, len(rois_p), figsize=(4.5 * len(rois_p), 5), sharey=True)
    if len(rois_p) == 1:
        axes = [axes]

    for ax, roi in zip(axes, rois_p):
        sub = perm_df[perm_df["roi"] == roi].copy()
        sub["label"] = sub["rule_a"].str[:4] + " vs " + sub["rule_b"].str[:4]
        sub = sub.sort_values("delta", ascending=True)
        ax.barh(
            range(len(sub)),
            sub["delta"],
            color=["#2ecc71" if s != "ns" else "#e74c3c" for s in sub["sig_fdr"]],
            alpha=0.8
        )
        ax.axvline(0, color="black", lw=0.8)
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(sub["label"], fontsize=7)
        ax.set_xlabel("Δρ (A−B)", fontsize=10)
        ax.set_title(f"ROI: {roi}", fontsize=11)

    plt.suptitle("Pairwise Δρ (green=FDR-sig, red=ns)", fontsize=12)
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / "permutation_forest_fdr.png"), dpi=150, bbox_inches="tight")
    plt.close()
    print("  permutation_forest_fdr.png saved")


Generating plots...
  rsa_comparison_cnn.png saved
  seed_variability.png saved
  permutation_forest_fdr.png saved


In [22]:
print("\n" + "="*55 + "\nSUMMARY\n" + "="*55)
print(f"  Seeds: {SEEDS}")
print(f"  THINGS input: {IMG_SIZE}×{IMG_SIZE}  |  Device: {DEVICE}")

if 'df' in globals() and not df.empty:
    pivot = df.pivot_table(index="roi", columns="rule", values="rho")
    print("\n  Mean RSA scores (across seeds):")
    print(pivot.round(4).to_string())

print(f"\nDone. Outputs: {OUTPUTS_DIR}")


SUMMARY
  Seeds: [42, 123, 456, 789, 1337]
  THINGS input: 224×224  |  Device: cpu

  Mean RSA scores (across seeds):
rule  Backprop  Feedback Alignment  Predictive Coding  Random Weights    STDP
roi                                                                          
IT      0.0132              0.0116             0.0136          0.0078  0.0119
LOC     0.0116              0.0056             0.0060         -0.0050  0.0058
V1      0.0004              0.0132             0.0608          0.0833  0.0673
V2     -0.0024              0.0060             0.0285          0.0495  0.0384

Done. Outputs: C:\Users\nilsl\Desktop\Projekte\learning-rules-rsa\outputs


In [25]:
# Einmalig: STDP Gewichte exportieren
torch.manual_seed(SEEDS[0])
np.random.seed(SEEDS[0])
random.seed(SEEDS[0])

model, _ = train_stdp(train_loader)

stdp_weights = {
    "conv1.weight": model.L1.conv.weight.data,
    "conv2.weight": model.L2.conv.weight.data,
    "conv3.weight": model.L3.conv.weight.data,
}
torch.save(stdp_weights, str(OUTPUTS_DIR / "model_weights_stdp.pt"))
print("Saved model_weights_stdp.pt")

    Epoch  10: loss=1.3426  acc=0.527
    Epoch  20: loss=1.1858  acc=0.579
    Epoch  30: loss=1.0981  acc=0.617
    Epoch  40: loss=1.0324  acc=0.635
Saved model_weights_stdp.pt


In [24]:
# In einer neuen Zelle:
print(dir(model.L1))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'conv', 'first_spike', 'forward', 'poisson_spikes', 'stdp_update']
